# Análisis interno de ESM-2: Proyecto Completo (Rubric Compliant)

Trabajo de Segundo Corte - Ciencia de Datos  
Universidad de Pamplona - Ingeniería de Sistemas

**Autor:** Anderson González (@Albonire)  
**Modelo:** `facebook/esm2_t6_8M_UR50D`

Este notebook integra todas las actividades de análisis siguiendo los requisitos técnicos de la guía:
1. Marco teórico detallado.
2. Ejecución y extracción de formas internas (Tokens, IDs, Shapes).
3. Inspección real del código fuente (fragmentos de Hugging Face).
4. Visualización de embeddings (Residuo vs Global) y PCA.
5. Análisis de matrices de atención (Capa temprana, profunda y cabezas).
6. Masked Language Modeling (MLM) con contexto bidireccional.
7. Usos reales y limitaciones documentadas.

> Nota: este trabajo no entrena el modelo. El objetivo es analizar un transformer científico preentrenado.

## 1. Marco teórico

### Transformer
El Transformer (Vaswani et al., 2017) reemplazó la recurrencia por el mecanismo de self-attention, permitiendo procesar secuencias en paralelo y capturar relaciones de largo alcance.

### Familias de transformers
- **Encoder-only** (BERT, ESM-2): atención bidireccional. Sirve para comprender y extraer embeddings.
- **Decoder-only** (GPT): atención causal. Sirve para generación autoregresiva.
- **Encoder-decoder** (T5): un encoder comprende y un decoder genera.

### Q, K, V y Multi-head Attention
La atención usa tres vectores por token: Query (lo que busca), Key (cómo se ofrece) y Value (información). La fórmula central es: $softmax(QK^T / \sqrt{d_k})V$.

In [ ]:
%pip install -q torch transformers matplotlib scikit-learn pandas tabulate

In [ ]:
import os
import inspect
import textwrap
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from transformers import AutoTokenizer, EsmModel, EsmForMaskedLM
from transformers.models.esm import modeling_esm

MODEL_NAME = "facebook/esm2_t6_8M_UR50D"
SEQUENCES = {
    "Original": "MKTAYIAKQRQISFVKSHFSRQDILD",
    "Mutada":   "MKTAFIAKQRQISFVKSHFSRQDILD",
    "Alterada": "DLIDQRSFHSSKVFSIQRQKAIYATKM",
}

os.makedirs("outputs", exist_ok=True)
print("PyTorch:", torch.__version__)

### Carga de Modelos e Inspección de Configuración

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
try:
    model = EsmModel.from_pretrained(MODEL_NAME, attn_implementation="eager")
except:
    model = EsmModel.from_pretrained(MODEL_NAME)
model.eval()

mlm_model = EsmForMaskedLM.from_pretrained(MODEL_NAME)
mlm_model.eval()

cfg = model.config
print(f"Capas: {cfg.num_hidden_layers} | Cabezas: {cfg.num_attention_heads} | Hidden Size: {cfg.hidden_size}")

## 2. Ejecución y formas internas

Reportamos longitud, tokens, IDs y formas de los tensores internos.

In [ ]:
def analyze_deep(name, seq):
    inputs = tokenizer(seq, return_tensors="pt")
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True, output_attentions=True)
    
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    ids = inputs["input_ids"][0].tolist()
    
    print(f"--- {name} ---")
    print(f"Longitud: {len(seq)} | Tokens: {len(tokens)}")
    print(f"IDs: {ids}")
    print(f"Hidden States: {len(out.hidden_states)} | Attentions: {len(out.attentions) if out.attentions else 'N/A'}")
    return inputs, out

results = {n: analyze_deep(n, s) for n, s in SEQUENCES.items()}

## 3. Inspección del código fuente

Extraemos fragmentos reales del código de ESM en Hugging Face para identificar proyecciones Q, K, V.

In [ ]:
def show_code(title, obj, lines=50):
    print(f"\n{'='*20} {title} {'='*20}")
    try:
        src = inspect.getsource(obj).splitlines()[:lines]
        print("\n".join(src))
    except: print("No disponible")

show_code("EsmSelfAttention (Q, K, V logic)", modeling_esm.EsmSelfAttention, 60)
show_code("EsmEmbeddings", modeling_esm.EsmEmbeddings, 40)

## 4. Embeddings y PCA

Comparamos la similitud entre secuencias y proyectamos los residuos con PCA.

In [ ]:
glob_embs = {}
all_res_vecs = []
labels = []

for n, (inp, out) in results.items():
    res_vecs = out.last_hidden_state[0, 1:-1, :].numpy()
    glob_embs[n] = res_vecs.mean(axis=0)
    all_res_vecs.append(res_vecs)
    labels.extend([n] * res_vecs.shape[0])

names = list(glob_embs.keys())
matrix = np.array([glob_embs[n] for n in names])
sim = cosine_similarity(matrix)
print("Matriz de Similitud Coseno:")
display(pd.DataFrame(sim, index=names, columns=names))

coords = PCA(n_components=2).fit_transform(np.vstack(all_res_vecs))
plt.figure(figsize=(8,6))
for n in names:
    idx = [i for i,l in enumerate(labels) if l==n]
    plt.scatter(coords[idx,0], coords[idx,1], label=n, alpha=0.6)
plt.title("PCA de Residuo-Embeddings")
plt.legend()
plt.savefig("outputs/pca_embeddings.png")
plt.show()

## 5. Matrices de atención

Comparamos la capa 0 (local) con la última capa (global) y dos cabezas distintas.

In [ ]:
def plot_att(out, layer, head, title, fname):
    if out.attentions:
        att = out.attentions[layer][0, head].detach().numpy()
        plt.imshow(att, cmap='viridis')
        plt.title(title)
        plt.colorbar()
        plt.savefig(f"outputs/{fname}")
        plt.show()

orig_out = results["Original"][1]
plot_att(orig_out, 0, 0, "Capa 0, Cabeza 0 (Local)", "att_early.png")
plot_att(orig_out, cfg.num_hidden_layers-1, 0, "Última Capa (Global)", "att_late.png")
plot_att(orig_out, 0, 1, "Capa 0, Cabeza 1 (Diferente foco)", "att_head1.png")

## 6. Masked Language Modeling

Ocultamos la Tirosina (Y) en la posición 4 y observamos la predicción bidireccional.

In [ ]:
masked = "MKTAYIAKQRQISFVKSHFSRQDILD".replace("Y", tokenizer.mask_token, 1)
inp_m = tokenizer(masked, return_tensors="pt")
m_idx = (inp_m.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
with torch.no_grad():
    logits = mlm_model(**inp_m).logits
top5 = torch.topk(logits[0, m_idx, :], 5)
for s, i in zip(top5.values[0], top5.indices[0]):
    print(f"Token: {tokenizer.convert_ids_to_tokens([i.item()])[0]} | Score: {s:.4f}")

## 7. Usos y Limitaciones

**Usos:** Búsqueda semántica, análisis de variantes, predicción 3D (ESMFold).
**Límites:** No es causalidad biológica, requiere validación experimental, sesgos de entrenamiento.